# Dijet reco-to-gen closures

This notebook follows `macro/plotMcClosures.C::plotDiJetClosures`: it projects the configured $p_T^{ave}$ interval, first compares the unnormalized full Reco, Ref, and Gen $\eta_{CM}^{dijet}$ distributions, then unit-normalizes the full distributions with ROOT `TH1::Scale`, and constructs unnormalized forward/backward ratios with ROOT `TH1::Divide`. Each lower panel is the corresponding Reco/Gen or Ref/Gen closure ratio.

Add another reco-like or smeared distribution by appending one `DijetClosureCurve` below. Its CM, Forward, and Backward histogram-key templates must share the same eta-cut index.

<!-- detailed-workflow-guide -->

### Detailed workflow and closure meaning

The closure ratio in bin $i$ is $C_i=N_i^{reco}/N_i^{gen}$, or the corresponding normalized-shape ratio when normalization is requested. A value near one means the selected reconstruction/smearing model reproduces that generator distribution within the displayed statistical precision. Independent reco and gen weighted histograms generally have correlated origins, although the displayed ROOT ratio uses the configured approximation.

Forward/backward closure is built from unnormalized eta projections as $F/B(|\eta|)=N(+|\eta|)/N(-|\eta|)$ with independent errors. Normalizing the full eta distribution before F/B is unnecessary because a common factor cancels. Empty denominators are undefined and must not be interpreted as physical zero ratios.

## Environment and imports

This notebook locates the repository dynamically and imports PyROOT from the
active project environment. Start Jupyter from the repository root with
`py-env/bin/python -m jupyter notebook`; no machine-specific ROOT paths are
added at runtime.


In [ ]:
# Cell role: initialize the reproducible Python/ROOT environment and shared helpers.
# Interpretation: No physics histogram is modified here; ROOT ownership is configured before files open.
# The preceding Markdown gives the equations and physics assumptions for this step.
%load_ext autoreload
%autoreload 2

from pathlib import Path
import os

import sys

# Locate the repository without relying on a machine-specific absolute path.
PROJECT_ROOT = next(
    (
        candidate
        for candidate in (Path.cwd(), *Path.cwd().parents)
        if (candidate / "CMakeLists.txt").is_file()
        and (candidate / "hist_analysis").is_dir()
    ),
    None,
)
if PROJECT_ROOT is None:
    raise RuntimeError(
        "Cannot locate the jetAnalysis repository. Start Jupyter from its root."
    )
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from hist_analysis.python.notebook_setup import load_root

# Batch mode keeps plots reproducible and sends them to notebook/output files.
ROOT = load_root(batch=True)

from hist_analysis.python.notebook_setup import load_root

# Batch mode keeps plots reproducible and sends them to notebook/output files.
ROOT = load_root(batch=True)

from hist_analysis.config.files import BASE_DIR
from hist_analysis.config.histograms import (
    DIJET_DELTA_PHI_SELECTION_LABEL, DIJET_PTAVE_BINS,
)
from hist_analysis.python.dijet_closures import (
    DijetClosureCurve, build_dijet_gen_comparisons,
)
from hist_analysis.python.histogram_io import (
    resolve_combined_file, resolve_direction_file,
)
from hist_analysis.python.plotting import draw_closure
from hist_analysis.python.root_style import (
    DEFAULT_PLOT_STYLE, draw_text_block, save_canvas, set_1d_style,
    set_legend_style, set_pad_style, style_single_panel_axes,
)


In [ ]:
# Cell role: perform analysis step 2.
# Interpretation: Operations use the binning, normalization, and uncertainty conventions documented above.
# The preceding Markdown gives the equations and physics assumptions for this step.
ROOT.gStyle.SetOptStat(0)
ROOT.gStyle.SetPalette(ROOT.kBird)
ROOT.TH1.AddDirectory(False)

## Configuration

The defaults compare nominal Gen, Reco, and matched Ref and select the standard 1.9 jet-eta cut by its canonical macro index. Set `ETA_CUT_INDICES = range(len(ETA_CUTS))` to process all seven cuts. The raw full-distribution closure is always built with `normalization='none'` before the separately configured normalized-shape closure. `NORMALIZATION = 'integral'` gives a per-bin unit-area shape; `bin_width` produces a unit-area density. `FORWARD_BACKWARD_RATIO_OPTION` must remain `''` because Forward and Backward are independent weighted populations. The later ratio of one already-constructed F/B histogram to another is configured separately with `FB_CLOSURE_RATIO_OPTION`; the user may select `''` or `'B'`.

In [ ]:
# Cell role: define and validate user-facing analysis configuration.
# Interpretation: Changing these values can change inputs, selections, binning, normalization, or outputs.
# The preceding Markdown gives the equations and physics assumptions for this step.
GENERATOR = 'embedding'       # embedding or pythia
DIRECTION = 'combined'        # pgoing, Pbgoing, or combined
FILE_STEM = 'jetId'
ETA_CUTS = (1.4, 1.5, 1.6, 1.7, 1.8, 1.9, 2.5)
ETA_CUT_INDICES = (5,)       # 1.9; use range(len(ETA_CUTS)) for all cuts
REBIN_ETA = 2
NORMALIZATION = 'integral'   # macro default: none, integral, or bin_width
FORWARD_BACKWARD_RATIO_OPTION = ''  # F / B: never use binomial errors
FULL_CLOSURE_RATIO_OPTION = 'B'      # use 'B' only for a true subset ratio
FB_CLOSURE_RATIO_OPTION = 'B'       # user choice for (F/B)_a / (F/B)_b: '' or 'B'
FULL_RATIO_RANGE = (0.75, 1.25)
RAW_FULL_RATIO_RANGE = (0.75, 1.25)
FB_RATIO_RANGE = (0.75, 1.3)
FB_DOUBLE_RATIO_RANGE = (0.75, 1.25)
SAVE_PNG = False
DRAW_GRID = True
OUTPUT_DIR = Path(os.environ.get(
    'DIJET_RECO_TO_GEN_OUTPUT_DIR',
    PROJECT_ROOT / 'hist_analysis' / 'output' / 'dijet_reco_to_gen_closures',
))

CURVES = (
    DijetClosureCurve(
        'Reco', 'hRecoDijetPtEtaCM_{eta_cut_index}',
        'hRecoDijetPtEtaForward_{eta_cut_index}',
        'hRecoDijetPtEtaBackward_{eta_cut_index}',
    ),
    DijetClosureCurve(
        'Gen', 'hGenDijetPtEtaCM_{eta_cut_index}',
        'hGenDijetPtEtaForward_{eta_cut_index}',
        'hGenDijetPtEtaBackward_{eta_cut_index}',
    ),
    DijetClosureCurve(
        'Ref', 'hRefDijetPtEtaCM_{eta_cut_index}',
        'hRefDijetPtEtaForward_{eta_cut_index}',
        'hRefDijetPtEtaBackward_{eta_cut_index}'),
    # DijetClosureCurve(
    #         'RefSel', 'hRefSelDijetPtEtaCM_{eta_cut_index}',
    #         'hRefSelDijetPtEtaForward_{eta_cut_index}',
    #         'hRefSelDijetPtEtaBackward_{eta_cut_index}'),
    # DijetClosureCurve('Reco JER x1.0', 'hRecoDijetPtEtaCMJerDef_{eta_cut_index}',
    #     'hRecoDijetPtEtaForwardJerDef_{eta_cut_index}',
    #     'hRecoDijetPtEtaBackwardJerDef_{eta_cut_index}'),
)

In [ ]:
# Cell role: define and validate user-facing analysis configuration.
# Interpretation: Changing these values can change inputs, selections, binning, normalization, or outputs.
# The preceding Markdown gives the equations and physics assumptions for this step.
def mc_file(generator, direction):
    if direction == 'combined':
        return resolve_combined_file(BASE_DIR, generator, FILE_STEM)
    return resolve_direction_file(BASE_DIR, generator, direction, FILE_STEM)

DIRECTION_LABELS = {
    'pgoing': 'p-going',
    'Pbgoing': 'Pb-going',
    'combined': 'Combined',
}
if DIRECTION not in DIRECTION_LABELS:
    raise ValueError(f'Unsupported DIRECTION={DIRECTION!r}')
DIRECTION_LABEL = DIRECTION_LABELS[DIRECTION]

INPUT_FILE = mc_file(GENERATOR, DIRECTION)
if not INPUT_FILE.exists():
    raise FileNotFoundError(f'Missing configured ROOT file: {INPUT_FILE}')
INPUT_FILE

## Build projections and ROOT ratios

The raw full distributions are projected and compared before any normalization. A second set of full distributions is then normalized according to `NORMALIZATION`. Forward and backward projections are deliberately not normalized before division. With the default `integral` mode, each normalized full eta shape has a unit sum of in-range bin contents; `bin_width` instead makes the bin-width-weighted integral equal to one.

In [ ]:
# Cell role: construct derived ratios, efficiencies, or correction factors.
# Interpretation: The numerator/denominator relationship determines whether independent or binomial errors are valid.
# The preceding Markdown gives the equations and physics assumptions for this step.
closure_results = {}

for eta_cut_index in ETA_CUT_INDICES:
    if eta_cut_index < 0 or eta_cut_index >= len(ETA_CUTS):
        raise IndexError(f'Invalid eta-cut index: {eta_cut_index}')
    eta_cut = ETA_CUTS[eta_cut_index]
    eta_x_range = (-eta_cut - 0.1, eta_cut + 0.1)
    fb_x_range = (0.0, eta_cut + 0.1)
    for ptave_range in DIJET_PTAVE_BINS:
        raw_eta_shapes, _, raw_keys = build_dijet_gen_comparisons(
            INPUT_FILE, CURVES, eta_cut_index=eta_cut_index,
            ptave_range=ptave_range, nominal='Gen',
            rebin_eta=REBIN_ETA, normalization='none',
            ratio_option=FORWARD_BACKWARD_RATIO_OPTION,
        )
        eta_shapes, fb_ratios, keys = build_dijet_gen_comparisons(
            INPUT_FILE, CURVES, eta_cut_index=eta_cut_index,
            ptave_range=ptave_range, nominal='Gen',
            rebin_eta=REBIN_ETA, normalization=NORMALIZATION,
            ratio_option=FORWARD_BACKWARD_RATIO_OPTION,
        )
        eta_cut_tag = int(round(10.0 * eta_cut))
        ptave_tag = f'{ptave_range[0]:g}_{ptave_range[1]:g}'.replace('.', 'p')
        selection_tag = f'{GENERATOR}_{DIRECTION}_etaCM_{eta_cut_tag}_ptave_{ptave_tag}'
        full_output_tag = (
            f'{GENERATOR}_{DIRECTION}_full_etaCM_{eta_cut_tag}_ptave_{ptave_tag}'
        )
        raw_full_output_tag = (
            f'{GENERATOR}_{DIRECTION}_full_raw_etaCM_{eta_cut_tag}_ptave_{ptave_tag}'
        )
        fb_output_tag = (
            f'{GENERATOR}_{DIRECTION}_fb_etaCM_{eta_cut_tag}_ptave_{ptave_tag}'
        )
        annotations = (
            GENERATOR.capitalize(),
            DIRECTION_LABEL,
            'CM frame',
            f'{ptave_range[0]:g} < p_{{T}}^{{ave}} < {ptave_range[1]:g} GeV',
            f'|#eta_{{CM}}^{{jet}}| < {eta_cut:g}',
            'p_{T}^{Lead} > 50 GeV',
            'p_{T}^{SubLead} > 40 GeV',
            DIJET_DELTA_PHI_SELECTION_LABEL,
        )

        eta_y_title = {
            'none': 'dN/d#eta_{CM}^{dijet}',
            'integral': '1/N dN/d#eta_{CM}^{dijet}',
            'bin_width': '1/N dN/d#eta_{CM}^{dijet}',
        }[NORMALIZATION]
        raw_eta_canvas, raw_eta_to_gen = draw_closure(
            raw_eta_shapes, 'Gen', title='',
            x_title='#eta_{CM}^{dijet}', y_title='dN/d#eta_{CM}^{dijet}',
            ratio_range=RAW_FULL_RATIO_RANGE, x_range=eta_x_range,
            annotations=annotations, grid=DRAW_GRID, headroom=1.6,
            draw_nominal_ratio=False, ratio_option=FULL_CLOSURE_RATIO_OPTION,
            style_indices={'Reco': 0, 'Gen': 1, 'Ref': 5},
            output=OUTPUT_DIR / f'{raw_full_output_tag}.pdf',
            save_png=SAVE_PNG, canvas_name=raw_full_output_tag,
        )
        eta_canvas, eta_to_gen = draw_closure(
            eta_shapes, 'Gen', title='',
            x_title='#eta_{CM}^{dijet}', y_title=eta_y_title,
            ratio_range=FULL_RATIO_RANGE, x_range=eta_x_range,
            annotations=annotations, grid=DRAW_GRID, headroom=1.6,
            draw_nominal_ratio=False, ratio_option=FULL_CLOSURE_RATIO_OPTION,
            style_indices={'Reco': 0, 'Gen': 1, 'Ref' : 5},
            output=OUTPUT_DIR / f'{full_output_tag}.pdf', save_png=SAVE_PNG,
            canvas_name=full_output_tag,
        )
        fb_canvas, fb_to_gen = draw_closure(
            fb_ratios, 'Gen', title='',
            x_title='#eta_{CM}^{dijet}', y_title='Forward / Backward',
            ratio_range=FB_DOUBLE_RATIO_RANGE, x_range=fb_x_range,
            y_range=FB_RATIO_RANGE,
            annotations=annotations, grid=DRAW_GRID, headroom=1.6,
            draw_nominal_ratio=False, ratio_option=FB_CLOSURE_RATIO_OPTION,
            style_indices={'Reco': 0, 'Gen': 1, 'Ref' : 5},
            output=OUTPUT_DIR / f'{fb_output_tag}.pdf',
            save_png=SAVE_PNG, canvas_name=fb_output_tag,
        )
        closure_results[selection_tag] = {
            'raw_eta_shapes': raw_eta_shapes,
            'raw_eta_to_gen': raw_eta_to_gen,
            'eta_shapes': eta_shapes, 'eta_to_gen': eta_to_gen,
            'forward_backward': fb_ratios, 'forward_backward_to_gen': fb_to_gen,
            'raw_eta_canvas': raw_eta_canvas, 'eta_canvas': eta_canvas,
            'forward_backward_canvas': fb_canvas,
            'raw_keys': raw_keys, 'keys': keys,
        }
        print(selection_tag, keys)
        display(raw_eta_canvas)

In [ ]:
# Cell role: construct derived ratios, efficiencies, or correction factors.
# Interpretation: The numerator/denominator relationship determines whether independent or binomial errors are valid.
# The preceding Markdown gives the equations and physics assumptions for this step.
# For each configured curve and pTave interval, compare F/B across eta cuts.
forward_backward_eta_cut_overlay_results = {}
if not ETA_CUT_INDICES:
    raise ValueError('ETA_CUT_INDICES must contain at least one eta-cut index')
invalid_eta_cut_indices = [
    index for index in ETA_CUT_INDICES if index < 0 or index >= len(ETA_CUTS)
]
if invalid_eta_cut_indices:
    raise IndexError(f'Invalid eta-cut indices: {invalid_eta_cut_indices}')
overlay_fb_x_range = (
    0.0, max(ETA_CUTS[index] for index in ETA_CUT_INDICES) + 0.1,
)

for curve in CURVES:
    curve_tag = ''.join(character.lower() if character.isalnum() else '_'
                        for character in curve.label).strip('_')
    for ptave_range in DIJET_PTAVE_BINS:
        ptave_tag = f'{ptave_range[0]:g}_{ptave_range[1]:g}'.replace('.', 'p')
        output_tag = (
            f'{GENERATOR}_{DIRECTION}_{curve_tag}_fb_etaCutOverlay_ptave_{ptave_tag}'
        )
        canvas = ROOT.TCanvas(
            f'c_{output_tag}', '',
            DEFAULT_PLOT_STYLE.canvas_width, DEFAULT_PLOT_STYLE.canvas_height,
        )
        set_pad_style(canvas, grid_x=DRAW_GRID, grid_y=DRAW_GRID)
        canvas.SetLeftMargin(DEFAULT_PLOT_STYLE.single_panel_left_margin)
        canvas.SetBottomMargin(DEFAULT_PLOT_STYLE.single_panel_bottom_margin)

        legend_height = 0.045 * len(ETA_CUT_INDICES)
        legend = ROOT.TLegend(0.66, 0.88 - legend_height, 0.88, 0.88)
        set_legend_style(legend)
        overlay_histograms = {}

        for style_index, eta_cut_index in enumerate(ETA_CUT_INDICES):
            eta_cut = ETA_CUTS[eta_cut_index]
            eta_cut_tag = int(round(10.0 * eta_cut))
            selection_tag = (
                f'{GENERATOR}_{DIRECTION}_etaCM_{eta_cut_tag}_ptave_{ptave_tag}'
            )
            source_histogram = closure_results[selection_tag]['forward_backward'][curve.label]
            histogram_name = (
                f'h_{output_tag}_etaCM_{eta_cut_tag}'
            )
            histogram = source_histogram.Clone(histogram_name)
            histogram.SetDirectory(0)
            histogram.SetTitle('')
            histogram.GetXaxis().SetTitle('#eta_{CM}^{dijet}')
            histogram.GetYaxis().SetTitle('Forward / Backward')
            histogram.GetXaxis().SetRangeUser(*overlay_fb_x_range)
            histogram.GetYaxis().SetRangeUser(*FB_RATIO_RANGE)
            set_1d_style(histogram, style_index)
            style_single_panel_axes(histogram)
            histogram.Draw('E1' if style_index == 0 else 'E1 SAME')
            legend.AddEntry(histogram, f'|#eta_{{CM}}^{{jet}}| < {eta_cut:g}', 'p')
            overlay_histograms[eta_cut_index] = histogram

        legend.Draw()
        annotations = draw_text_block(canvas, (
            GENERATOR.capitalize(),
            f'{curve.label} dijets',
            'CM frame',
            f'{ptave_range[0]:g} < p_{{T}}^{{ave}} < {ptave_range[1]:g} GeV',
            'p_{T}^{Lead} > 50 GeV',
            'p_{T}^{SubLead} > 40 GeV',
            DIJET_DELTA_PHI_SELECTION_LABEL,
        ))
        canvas.Modified()
        canvas.Update()
        save_canvas(
            canvas, OUTPUT_DIR / f'{output_tag}.pdf', save_png=SAVE_PNG,
        )
        canvas._forward_backward_eta_cut_overlay_objects = [
            legend, *annotations, *overlay_histograms.values(),
        ]
        result_key = (curve.label, ptave_range)
        forward_backward_eta_cut_overlay_results[result_key] = {
            'canvas': canvas, 'histograms': overlay_histograms,
        }
        display(canvas)

## Inspect numerical results

The retained dictionaries contain the raw and normalized ROOT histograms, closure ratios, and canvases for interactive inspection. This cell reports each raw yield, the normalized integrals, and the extrema of the raw-full, normalized-full, and F/B closure ratios to Gen.

In [ ]:
# Cell role: construct derived ratios, efficiencies, or correction factors.
# Interpretation: The numerator/denominator relationship determines whether independent or binomial errors are valid.
# The preceding Markdown gives the equations and physics assumptions for this step.
for tag, result in closure_results.items():
    print(f'\n{tag}')
    for label in (curve.label for curve in CURVES if curve.label != 'Gen'):
        raw_eta_ratio = result['raw_eta_to_gen'][label]
        eta_ratio = result['eta_to_gen'][label]
        fb_ratio = result['forward_backward_to_gen'][label]
        eta_values = [eta_ratio.GetBinContent(i) for i in range(1, eta_ratio.GetNbinsX() + 1)
                      if eta_ratio.GetBinContent(i) != 0.0]
        raw_eta_values = [
            raw_eta_ratio.GetBinContent(i)
            for i in range(1, raw_eta_ratio.GetNbinsX() + 1)
            if raw_eta_ratio.GetBinContent(i) != 0.0
        ]
        fb_values = [fb_ratio.GetBinContent(i) for i in range(1, fb_ratio.GetNbinsX() + 1)
                     if fb_ratio.GetBinContent(i) != 0.0]
        print(label, {
            'raw_yield': result['raw_eta_shapes'][label].Integral(),
            'raw/gen range': (min(raw_eta_values), max(raw_eta_values))
                             if raw_eta_values else None,
            'bin_sum': result['eta_shapes'][label].Integral(),
            'width_integral': result['eta_shapes'][label].Integral('width'),
            'eta/gen range': (min(eta_values), max(eta_values)) if eta_values else None,
            '(F/B)/(F/B)_gen range': (min(fb_values), max(fb_values)) if fb_values else None,
        })